# Untold Lores — Image + Animation Server
Runs FLUX.1-schnell (text→image) and SVD (image→video) as a local API.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order
3. Copy the `LOCAL_MODEL_URL=...` line into your `.env` file
4. Run `npm run test-video -- --reuse-story` on your laptop

In [ ]:
# Cell 1 — Install dependencies
!pip install -q diffusers transformers accelerate flask pyngrok imageio[ffmpeg] sentencepiece protobuf

In [ ]:
# Cell 2 — Load models (takes 10-20 min first time, cached after)
import torch
from diffusers import FluxPipeline, StableVideoDiffusionPipeline
from diffusers.utils import export_to_video
from PIL import Image
import io, base64, os, tempfile

print('🔧 Loading FLUX.1-schnell...')
flux_pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell',
    torch_dtype=torch.bfloat16
)
flux_pipe.enable_model_cpu_offload()
print('✅ FLUX ready!')

print('🔧 Loading Stable Video Diffusion...')
svd_pipe = StableVideoDiffusionPipeline.from_pretrained(
    'stabilityai/stable-video-diffusion-img2vid-xt',
    torch_dtype=torch.float16,
    variant='fp16'
)
svd_pipe.enable_model_cpu_offload()
print('✅ SVD ready!')

In [ ]:
# Cell 3 — Define API server
from flask import Flask, request, send_file, jsonify
import threading

app = Flask(__name__)
TMP_DIR = tempfile.mkdtemp()

@app.route('/health')
def health():
    return jsonify({'status': 'ok'})

@app.route('/txt2img', methods=['POST'])
def txt2img():
    data = request.json
    prompt   = data.get('prompt', 'cinematic scene')
    seed     = int(data.get('seed', 42))
    width    = int(data.get('width', 1280))
    height   = int(data.get('height', 720))

    generator = torch.Generator().manual_seed(seed)
    result = flux_pipe(
        prompt=prompt,
        width=width,
        height=height,
        guidance_scale=0.0,
        num_inference_steps=4,
        generator=generator,
    )
    image = result.images[0]

    buf = io.BytesIO()
    image.save(buf, format='JPEG', quality=90)
    buf.seek(0)
    return send_file(buf, mimetype='image/jpeg')

@app.route('/img2vid', methods=['POST'])
def img2vid():
    data = request.json
    image_b64 = data.get('image')
    seed      = int(data.get('seed', 42))

    # Decode and resize to SVD optimal resolution
    image_bytes = base64.b64decode(image_b64)
    image = Image.open(io.BytesIO(image_bytes)).convert('RGB').resize((1024, 576))

    generator = torch.Generator().manual_seed(seed)
    output = svd_pipe(
        image,
        num_frames=25,
        num_inference_steps=25,
        fps=6,
        motion_bucket_id=127,
        noise_aug_strength=0.02,
        generator=generator,
    )
    frames = output.frames[0]

    out_path = os.path.join(TMP_DIR, f'clip_{seed}.mp4')
    export_to_video(frames, out_path, fps=6)
    return send_file(out_path, mimetype='video/mp4')

# Start Flask in background thread
t = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False, threaded=False))
t.daemon = True
t.start()
print('✅ Flask server started on port 5000')

In [ ]:
# Cell 4 — Expose with ngrok
# Get your free token at https://ngrok.com (sign up → Your Authtoken)
from pyngrok import ngrok

NGROK_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'  # <-- replace this

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000).public_url

print('=' * 60)
print(f'🚀 Server is live!')
print(f'')
print(f'Add this to your .env file:')
print(f'LOCAL_MODEL_URL={public_url}')
print('=' * 60)

In [ ]:
# Cell 5 — Keep session alive (run this, leave it running)
# Colab disconnects after ~90 min of inactivity. This cell keeps it alive.
import time
print('⏳ Keeping session alive... (interrupt kernel when done generating)')
while True:
    time.sleep(60)
    print('.', end='', flush=True)